# A categorically organized CAS in Lean

This notebook is the acceptance proof of the CasDsl vertical slice: a
computer algebra system whose **user-facing interfaces are organized by the
mathematical categories where operations first make sense** — not by
implementation classes, and not by backends.

Everything below runs in a persistent Lean 4 kernel
([lean-jupyter-kernel](https://github.com/dzackgarza/lean-jupyter-kernel)).
Three invariants to watch for:

1. **Backend-blind syntax.** You will never see a backend named in an
   expression. Some results below are computed by SageMath through a direct
   typed adapter — the *developer's* routing configuration decides that,
   and `#explain_route` will show it. The mathematics doesn't change.
2. **Category-owned methods.** `factor`, `det`, `annihilator`, `nth` are
   declared on categories; objects receive them by membership and by
   *subcategory inheritance*, never by forwarding code on a leaf class.
3. **Semantic availability ≠ computability.** A method that makes
   mathematical sense stays available even when no implementation route
   exists yet — execution then fails with a *structured capability gap*
   (an auditable developer backlog item), never a fake value and never a
   type error. The final cell demonstrates this deliberately.


## 1 · Trusted arithmetic and assertions

`assert` is an *operational* assertion in the ordinary CAS sense: the
predicate is computed and trusted, with a fourfold outcome
`true | false | unknown | error`. Only `true` lets the cell commit.
No Lean theorem is generated, and no certificate is required — this is a
CAS, not a proof obligation machine.


In [1]:
assert 2 + 3 = 5

Starting Lean worker (/home/dzack/gitclones/lean-cas-dsl)…


1:0: ✓ 2 + 3 = 5


In [2]:
assert 2 + 3 = 0 in ℤ/5

1:0: ✓ 2 + 3 = 0 in ℤ/5


## 2 · Backend-blind factorization

`factor` is declared on the category of factorization-domain elements.
An integer receives it because `EuclideanElems(ℤ) ≤ FactorizationElems(ℤ)`
— the method arrives by **subcategory inheritance through the category
graph**, and is executed by whatever implementation the developer routed.


In [3]:
let n := 360 in ℤ

1:0: n := 360 ∈ ℤ


In [4]:
n.factor()

1:0: 2^3 * 3^2 * 5


2^3 * 3^2 * 5

The expression above never mentioned a backend. The routing that chose one
is developer diagnostics, not mathematics:


In [5]:
#explain_route n.factor()

1:0:   method:        factor
  receiver:      360 ∈ ℤ
  profile entry: EuclideanElems(ℤ)
  availability:  inherited through EuclideanElems(ℤ) ≤ FactorizationElems
  route:         backend sage, op "factor_int", priority 0
  pattern:       element of ℤ


  method:        factor
  receiver:      360 ∈ ℤ
  profile entry: EuclideanElems(ℤ)
  availability:  inherited through EuclideanElems(ℤ) ≤ FactorizationElems
  route:         backend sage, op "factor_int", priority 0
  pattern:       element of ℤ

`gcd` is declared on the same category as `factor` — greatest common
divisors exist in every unique factorization domain, which is where the
operation first makes sense. SPEC.md writes it in *prefix* position, and
that is exactly what it is: `gcd(84, 30)` **is** the method call
`84.gcd(30)`, resolved and routed identically. The prefix reading applies
only to a name that is unbound *and* declared as a method somewhere, so it
can never shadow a binding or invent an operation.

In [6]:
assert gcd(84, 30) = 6

1:0: ✓ gcd(84, 30) = 6


## 3 · Polynomials, canonical maps, and calling a polynomial

`ℤ ⊆ ℚ` denotes the preferred canonical map — so `map p to ℚ[x]` moves a
polynomial along it without ceremony. `factor` is routed where `p` lives:
ℤ[x] is a UFD, and comparing its factorization with the one in ℚ[x]
below — content and units differ in general — is itself instructive. And a polynomial can simply be
**called**: elaboration inserts evaluation through the preferred compatible
coefficient map. The mathematician writes `q(1)`, as on paper.

`map e to D` means: apply the preferred canonical map into `D` when one is
registered, and fail honestly otherwise. Canonical maps are *preferred
choices*, not necessarily injections — the inclusions `ℕ ⊆ ℤ ⊆ ℚ` are
monomorphisms, while `ℤ → ℤ/n` is the ring quotient, supplied by its
universal property. As of round two these are **registry data**: the
prelude registers them, the engine knows none of these facts, and an
unregistered pair fails with the honest `there is no preferred canonical
map` error — a missing coercion is never widened to a "reasonable"
conversion.

In [7]:
let p(x) := x^3 - 2x + 1 in ℤ[x]

1:0: p := x^3 - 2x + 1 ∈ ℤ[x]


Two membership questions about the ring `p` was just defined in. The second
is ordinary — `p` is a value, `ℤ[x]` is a set. The first is the interesting
one: `x` is *not* a session binding (defining `p` published nothing), but in
`x ∈ ℤ[x]` the only element of that ring a repeated `x` could name is its
indeterminate, so the assertion reads it that way — **locally**, for this
one line. Outside it, a bare `x` is still the honest `not bound` error.

In [8]:
assert x ∈ ℤ[x]

1:0: ✓ x ∈ ℤ[x]


In [9]:
assert p ∈ ℤ[x]

1:0: ✓ p ∈ ℤ[x]


`deg` is declared on `PolynomialElems`, a category `p` inhabits
*independently* of the factorization hierarchy: degree is a structural read
of a polynomial and makes sense over coefficient rings where factorization
does not. Neither membership implies the other.

In [10]:
p.deg()

1:0: 3


3

In [11]:
p.factor()

1:0: (x - 1) * (x^2 + x - 1)


(x - 1) * (x^2 + x - 1)

In [12]:
let q := map p to ℚ[x]

1:0: q := x^3 - 2x + 1 ∈ ℚ[x]


In [13]:
q.factor()

1:0: (x - 1) * (x^2 + x - 1)


(x - 1) * (x^2 + x - 1)

In [14]:
assert q(1) = 0

1:0: ✓ q(1) = 0


`roots` answers in the polynomial's **own coefficient ring** — that is the
mathematical content of the method, not a limitation of the backend. Over
ℤ, `x³ − 2x + 1` has the root `1`:

In [15]:
p.roots()

1:0: {1}


{1}

In [16]:
assert 1 ∈ p.roots()

1:0: ✓ 1 ∈ p.roots()


And SPEC.md's `x² − 2` over ℚ has **none**. The empty set is the answer —
not an error, and not a silent reach into an extension field. (`√2` and the
comparison `q.roots() ⊆ ℂ − ℚ` need ℂ, which this slice does not have yet.)

In [17]:
let q := x ↦ x² - 2 in ℚ[x]

1:0: q := x^2 - 2 ∈ ℚ[x]


In [18]:
q.roots()

1:0: {}


{}

The quotient `ℤ → ℤ/n` is one registered rule for *every* modulus: an
integer names its residue class. `n` is still `360`, and `360 ≡ 3 (mod 7)`:

In [19]:
map n to ℤ/7

1:0: 3


3

The coercion surface is itself auditable: these are the *only* maps
the surface will ever insert, straight from the registry (#9):

In [20]:
#canonical_maps

1:0:   map         op
  ℕ → ℤ       identity
              ℕ ⊆ ℤ: an element of ℕ already IS an integer (they share the `Value.int` representation), so the injection moves no data
  ℕ → ℚ       intToRat
              ℕ ⊆ ℚ: the composite of ℕ ⊆ ℤ ⊆ ℚ, registered explicitly because the coercion layer takes ONE hop (it does not compose rules)
  ℤ → ℚ       intToRat
              ℤ ⊆ ℚ: the fraction field of ℤ — the notebook's `map p to ℚ[x]` is this rule applied coefficient-wise
  ℤ → ℤ/_     intToMod
              ℤ → ℤ/n for EVERY modulus n (one rule, by `anyMod`): the ring quotient — an integer naming its residue class, which is what an ascription such as `let x := 7 in ℤ/5` inserts


  map         op
  ℕ → ℤ       identity
              ℕ ⊆ ℤ: an element of ℕ already IS an integer (they share the `Value.int` representation), so the injection moves no data
  ℕ → ℚ       intToRat
              ℕ ⊆ ℚ: the composite of ℕ ⊆ ℤ ⊆ ℚ, registered explicitly because the coercion layer takes ONE hop (it does not compose rules)
  ℤ → ℚ       intToRat
              ℤ ⊆ ℚ: the fraction field of ℤ — the notebook's `map p to ℚ[x]` is this rule applied coefficient-wise
  ℤ → ℤ/_     intToMod
              ℤ → ℤ/n for EVERY modulus n (one rule, by `anyMod`): the ring quotient — an integer naming its residue class, which is what an ascription such as `let x := 7 in ℤ/5` inserts

## 4 · Exact matrix algebra

Matrix literals use row-semicolon syntax; `det` and `inverse` are methods
of the square-matrix category, computed exactly over `ℚ`.


In [21]:
let M := [1, 2; 3, 4] in Mat₂(ℚ)

1:0: M := [1, 2; 3, 4] ∈ Mat₂(ℚ)


In [22]:
M.inverse()

1:0: [-2, 1; 3/2, -1/2]


[-2, 1; 3/2, -1/2]

In [23]:
assert M.det() = -2

1:0: ✓ M.det() = -2


## 5 · Subcategory inheritance, for real

`annihilator : Modules(ℤ) → Ideals(ℤ)` is declared **once**, on the parent
category. `F` below is declared in the *proper subcategory*
`SmallModules(ℤ)` — which contains **no forwarding declaration**. The
method arrives purely through the registered inclusion
`SmallModules ≤ Modules`. (The ascription is doing real semantic work:
`ℤ/4` *in a module category* means the ℤ-module ℤ/4, not the ring.)


In [24]:
let F := ℤ/4 in SmallModules(ℤ)

1:0: F := ℤ/4 as ℤ-module


In [25]:
F.annihilator()

1:0: (4)


(4)

## 6 · Transport along preferred functors

`cardinality` is declared on `Sets` — and a module is not a set. But the
prelude registers the forgetful functor `UnderlyingSet : Modules(ℤ) → Sets`
as *preferred*, so the resolver transports the **receiver**:
`F.cardinality()` resolves as `UnderlyingSet(F).cardinality()`. Nothing was
declared on modules, no forwarding method exists anywhere, and the ordinary
call syntax is unchanged.

In [26]:
F.cardinality()

1:0: 4


4

Membership transports the same way — `2` names a residue class of the
underlying set:

In [27]:
assert 2 ∈ F

1:0: ✓ 2 ∈ F


Equality, by contrast, is **category-bound**. `U(F) = {0, 1, 2, 3}` in
Sets — but `F` itself is a module, and there is no *unique* module
structure on that set, so bare `=` between objects of different
categories is trivially false: it never inserts the functor. Comparing
them requires explicitly asking the question in a common comparison
category — which is exactly what the Sets method `set_eq` does (its
receiver transports, like `∈` above):

In [28]:
assert F ≠ {0, 1, 2, 3}

1:0: ✓ F ≠ {0, 1, 2, 3}


In [29]:
F.set_eq({0, 1, 2, 3})

1:0: true


true

Two guarantees, both machine-checked in the build:

- transport runs **only where direct resolution finds nothing** —
  `annihilator` above still arrives untransported through
  `SmallModules(ℤ) ≤ Modules(ℤ)`, so registering a functor can never take
  a method away from an object that already had it;
- two applicable functors would be an honest *ambiguity error* naming both,
  never a silent pick.

The transport step itself is developer diagnostics, not mathematics:

In [30]:
#explain_route F.cardinality()

1:0:   method:        cardinality
  receiver:      ℤ/4 as ℤ-module
  transport:     functor UnderlyingSet : Modules → Sets
  image:         {0, 1, 2, 3}
  profile entry: FiniteSets(ℤ/4)
  availability:  inherited through FiniteSets(ℤ/4) ≤ CountableSets ≤ Sets
  route:         backend native, op "cardinality", priority 0
  pattern:       any set


  method:        cardinality
  receiver:      ℤ/4 as ℤ-module
  transport:     functor UnderlyingSet : Modules → Sets
  image:         {0, 1, 2, 3}
  profile entry: FiniteSets(ℤ/4)
  availability:  inherited through FiniteSets(ℤ/4) ≤ CountableSets ≤ Sets
  route:         backend native, op "cardinality", priority 0
  pattern:       any set

## 7 · Countable sets, ellipses, and indexing

Countability is mathematical structure — a monomorphism into ℕ — not a
backend capability. A *registered enumeration choice* labels elements, so
countable objects support `nth` (`X[k]`, 0-based) and `cardinality`.
Ellipsis literals are the exact Haskell-style progressions, nothing more.

The registered convention for `ℤ` is `0, 1, −1, 2, −2, …` — a documented,
revisitable choice, never a claim that ℤ is intrinsically ordered that way.


In [31]:
let X := {0, 1, 2, ...}

1:0: X := {0, 1, ...}


In [32]:
assert X = ℕ

1:0: ✓ X = ℕ


In [33]:
let Y := {0, 2, 4, ...}

1:0: Y := {0, 2, ...}


In [34]:
assert 8 ∈ Y

1:0: ✓ 8 ∈ Y


In [35]:
assert 9 ∉ Y

1:0: ✓ 9 ∉ Y


In [36]:
ℤ[3]

1:0: 2


2

`ℚ` indexes by *its* registered convention too — the Cantor zigzag
(`0, 1, −1, 1/2, −1/2, 2, −2, 1/3, …`, reduced fractions only), a
documented revisitable choice exactly like ℤ's (round three, #17):

In [37]:
ℚ[3]

1:0: 1/2


1/2

In [38]:
X.cardinality()

1:0: ℵ₀


ℵ₀

## 8 · Functions

A function is `binder ↦ body` together with the domains it runs between, and
the two spellings SPEC.md uses — the lambda and `f(t) := …` — denote the
*same* function. `ℝ → ℝ` is an ascription **domain tag** at this stage: it
says where the function is declared and attaches no analysis semantics.

Bodies are exact polynomials, which is what lets the assertions below be
identities of function *expressions* rather than samples at a few points.
Non-polynomial bodies (`t ↦ sin(t)`, `t ↦ e^t`) are an honest gap until the
calculus sections land — they are refused at the binding, never
approximated.

In [39]:
let h := t ↦ t² + 1 in ℝ → ℝ

1:0: h := t ↦ t^2 + 1 ∈ ℝ → ℝ


In [40]:
let hp(t) := t^2 + 1 in R->R

1:0: hp := t ↦ t^2 + 1 ∈ ℝ → ℝ


In [41]:
assert h = hp

1:0: ✓ h = hp


In [42]:
assert h(0) = 1

1:0: ✓ h(0) = 1


In [43]:
assert h(3) = 10

1:0: ✓ h(3) = 10


`h(-t) = h(t)` is not two numeric samples that happened to agree: both
sides *substitute* a polynomial into the body, and the normal forms are
compared. A function in scope makes the binder it names available as that
indeterminate, which is how `t` can be written freely here.

In [44]:
assert h(-t) = h(t)

1:0: ✓ h(-t) = h(t)


The leading-ascription spelling declares the same thing. `e` returns in §10,
where its image is a set.

In [45]:
let e: ℕ → ℕ := n ↦ 2n

1:0: e := n ↦ 2n ∈ ℕ → ℕ


Composition is the point of having functions at all: `f ∘ g` is a function
like any other, and the identity it satisfies is decided by substituting one
body into the other.

In [46]:
let f(t) = t^2 in RR->RR

1:0: f := t ↦ t^2 ∈ ℝ → ℝ


In [47]:
let g(t) = t^3 in RR->RR

1:0: g := t ↦ t^3 ∈ ℝ → ℝ


In [48]:
assert (f ∘ g)(t) = t^6

1:0: ✓ (f ∘ g)(t) = t^6


## 9 · Finite sets and set algebra

The set operations are *category-owned methods on `Sets`*, exactly like
`cardinality` and `contains`: `A ∪ B` **is** `A.union(B)`, resolved and
routed like any other call. The ascription says where the set lives —
`𝒫(ℤ)` and `2^ℤ` are the two spellings SPEC.md uses for the same powerset,
and ascribing to one is a *checked membership judgment* (`A ⊆ ℤ`), not an
annotation.

In [49]:
let A := {1, 2, 3} in 𝒫(ℤ)

1:0: A := {1, 2, 3}


In [50]:
let B := {3, 4, 5} in 2^ℤ

1:0: B := {3, 4, 5}


In [51]:
assert A ∪ B = {1, 2, 3, 4, 5}

1:0: ✓ A ∪ B = {1, 2, 3, 4, 5}


In [52]:
assert A ∩ B = {3}

1:0: ✓ A ∩ B = {3}


In [53]:
assert A \ B = {1, 2}

1:0: ✓ A \ B = {1, 2}


In [54]:
assert A △ B = {1, 2, 4, 5}

1:0: ✓ A △ B = {1, 2, 4, 5}


`|A|` is the cardinality method under another spelling. The interesting
cases are the two sets that are **denoted rather than listed**: the elements
of `A × B` are pairs and the elements of `𝒫(A)` are sets, and this slice's
`Value` presents neither — so both are *presentations*, like `{0, 2, 4, ...}`
is. Their cardinalities are exact cardinal arithmetic, which is what makes
`|𝒫(A)| = 2^|A|` a computed identity rather than a definition.

In [55]:
|A|

1:0: 3


3

In [56]:
A × B

1:0: {1, 2, 3} × {3, 4, 5}


{1, 2, 3} × {3, 4, 5}

In [57]:
assert |A × B| = 9

1:0: ✓ |A × B| = 9


In [58]:
assert |𝒫(A)| = 2^|A|

1:0: ✓ |𝒫(A)| = 2^|A|


Membership and inclusion are one decision procedure with two spellings:
`X ∈ 𝒫(A)` asks exactly what `X ⊆ A` asks. (SPEC.md also writes the ASCII
`in` for `∈`.)

The routes tell the usual second story. `∪ ∩ \ △` are *meaningful* for every
set — they are declared on `Sets` — but only explicit finite receivers are
routed, so `ℤ ∪ A` resolves and then reports a structured gap. `𝒫(ℕ)` is
uncountable, and this slice's `Cardinality` says it cannot state that size
rather than inventing `ℵ₀`. Both appear in the audit at the end.

In [59]:
assert 2 ∈ A

1:0: ✓ 2 ∈ A


In [60]:
assert 4 ∉ A

1:0: ✓ 4 ∉ A


In [61]:
assert A ⊆ A ∪ B

1:0: ✓ A ⊆ A ∪ B


In [62]:
assert A in 𝒫(ℤ)

1:0: ✓ A ∈ 𝒫(ℤ)


## 10 · Set comprehensions

`{n ∈ ℤ | n² ≤ 20}` is a claim about *all* integers, so it is **decided**,
never sampled. The guard is rewritten as `p(n) ⋈ 0` for an exact polynomial;
a Cauchy-style bound puts every root of `p` inside `±N`, so `p` keeps one
sign on each tail; evaluating it there says whether the tail satisfies the
guard. A satisfied tail means the set is infinite and the comprehension says
so — otherwise every solution lies inside the bound and each candidate is
tested exactly. There is no enumeration cutoff anywhere in that.

In [63]:
let S := {n ∈ ℤ | n² ≤ 20}

1:0: S := {-4, -3, -2, -1, 0, 1, 2, 3, 4}


In [64]:
assert S = {-4, -3, -2, -1, 0, 1, 2, 3, 4}

1:0: ✓ S = {-4, -3, -2, -1, 0, 1, 2, 3, 4}


In [65]:
assert |S| = 9

1:0: ✓ |S| = 9


An infinite comprehension is presented when its image *is* a presentation
the slice already has. `{2n | n ∈ ℕ}` is the arithmetic progression
`{0, 2, 4, ...}` — SPEC.md's own identity — so membership in it is **solved**
rather than searched: `10¹² ∈ E` costs exactly what `8 ∈ E` costs.

In [66]:
let E := {2n | n ∈ ℕ}

1:0: E := {0, 2, ...}


In [67]:
assert 8 ∈ E

1:0: ✓ 8 ∈ E


In [68]:
assert 9 ∉ E

1:0: ✓ 9 ∉ E


In [69]:
assert |E| = ℵ₀

1:0: ✓ |E| = ℵ₀


In [70]:
assert 1000000000000 ∈ E

1:0: ✓ 1000000000000 ∈ E


`e` was declared in §8 as `n ↦ 2n`. Its **image** is that same set, under
either spelling: `e.image()` is the one method functions own, and `e(ℕ)` —
applying a function to its source — is the same call. A guard bounds the
binder to a finite range, which is what turns the last SPEC.md line into an
explicit list.

In [71]:
assert e(ℕ) = E

1:0: ✓ e(ℕ) = E


In [72]:
e.image()

1:0: {0, 2, ...}


{0, 2, ...}

In [73]:
{e(n) | n ∈ ℕ, 0 ≤ n < 6}

1:0: {0, 2, 4, 6, 8, 10}


{0, 2, 4, 6, 8, 10}

`is_prime` is declared where primes first make sense — the elements of a
unique factorization domain — and routed for ℤ. Primality here is the
*normalized* one: `−7` is irreducible, but `7` is the normalized
representative of its associate class, so `−7` answers false.

**Disclosed gaps — the two SPEC.md §Ellipses comprehension lines, which fail
differently.**

`{n in ℕ | n.is_prime()}` *parses* and is refused **at the binding**: this
slice decides comprehensions whose guard is a polynomial comparison in the
binder, and a primality test is not one. No sampled membership, no truncated
enumeration, no wrong verdict — and the method itself works fine outside a
comprehension, as the cell below shows.

`{n in ℕ | f(n) ∈ 2ℕ}` does not even parse: `∈` is an assertion relation, not
a term operator, so the guard position rejects it and the statement splitter
runs what is left as fragments. `2ℕ` compounds it — there is no scaling
production, so it reads as two juxtaposed terms, which is why SPEC.md's
`assert Y = 2ℕ` would assert `Y = 2`. Both readings are the splitter's, not
this surface's: decided answers to *different* claims, which is exactly why
they are listed as gaps rather than trusted.

In [74]:
let m7 := 7 in ℤ

1:0: m7 := 7 ∈ ℤ


In [75]:
m7.is_prime()

1:0: true


true

## 11 · Semantic availability is not computability

`det` makes sense for a square matrix over any commutative ring — the
category layer says so (`MatrixElems`), and no implementation hole is
allowed to redefine the mathematics. But the developer has only routed
matrices with entries in ℚ: over the field ℤ/5, `det` is *semantically*
available and not yet executable. `gcd` is in the same position outside
ℤ — gcds exist in every UFD, and only the ℤ route is registered. The
audit surface shows every such hole as structured backlog:

In [76]:
#capability_gaps

1:0:   representative    method        category              status
  ℤ                 union         CountableSets(ℤ)      no route matches this presentation (1 registered for the method, none matching)
                                  available by: inherited through CountableSets(ℤ) ≤ Sets
  ℤ                 intersect     CountableSets(ℤ)      no route matches this presentation (1 registered for the method, none matching)
                                  available by: inherited through CountableSets(ℤ) ≤ Sets
  ℤ                 diff          CountableSets(ℤ)      no route matches this presentation (1 registered for the method, none matching)
                                  available by: inherited through CountableSets(ℤ) ≤ Sets
  ℤ                 symdiff       CountableSets(ℤ)      no route matches this presentation (1 registered for the method, none matching)
                                  available by: inherited through CountableSets(ℤ) ≤ Sets
  ℚ                 union   

  representative    method        category              status
  ℤ                 union         CountableSets(ℤ)      no route matches this presentation (1 registered for the method, none matching)
                                  available by: inherited through CountableSets(ℤ) ≤ Sets
  ℤ                 intersect     CountableSets(ℤ)      no route matches this presentation (1 registered for the method, none matching)
                                  available by: inherited through CountableSets(ℤ) ≤ Sets
  ℤ                 diff          CountableSets(ℤ)      no route matches this presentation (1 registered for the method, none matching)
                                  available by: inherited through CountableSets(ℤ) ≤ Sets
  ℤ                 symdiff       CountableSets(ℤ)      no route matches this presentation (1 registered for the method, none matching)
                                  available by: inherited through CountableSets(ℤ) ≤ Sets
  ℚ                 union        

In [77]:
let A := [1, 2; 3, 4] in Mat₂(ℤ/5)

1:0: A := [1, 2; 3, 4] ∈ Mat₂(ℤ/5)


So the next cell **fails on purpose** — with a structured
`NoImplementation` capability gap naming the method, the receiver
category, the presentation, and the routes considered. Not a parse error,
not a type error, and not a silent lie. This failing cell is part of the
proof.


In [78]:
A.det()

LeanError: NoImplementation: 'det' is mathematically available here, but no registered route can execute it for this presentation.
  method:            det
  receiver category: MatrixElems(2, ℤ/5)
  presentation:      [1, 2; 3, 4] ∈ Mat₂(ℤ/5)
  semantic path:     declared directly on MatrixElems(2, ℤ/5)
  routes considered: 1
    - det for element of Mat(_, ℚ) → backend sage, op "mat_det_q", priority 0
This is a developer backlog item, not a narrowing of the mathematics: the method stays available on the category.